In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import(
    accuracy_score,classification_report,confusion_matrix
)
from scipy.stats import randint

In [ ]:
df = pd.read_csv("bank_marketing.csv")
df.head()

In [ ]:
print(df.shape)
# print columns name 
print(df.columns.tolist)

In [ ]:
df.columns = df.columns.str.strip()
print(df.columns.tolist)

In [ ]:
target = "y"

# check missing values 
print(df.isnull().sum())

In [ ]:
# check data type 
df.dtypes

In [ ]:
print(df[target].value_counts())

# label encoding
encoder = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = encoder.fit_transform(df[col])

In [ ]:
# feature and target
X = df.drop(columns=[target])
y = df[target]
print("feature shape = ",X.shape)
print("Target Shape = ",y.shape)

In [ ]:
X_train , X_test , y_train ,y_test = train_test_split(X,y,test_size=0.20,random_state=42)
print("Traning ",X_train.shape)
print("Testing ",X_test.shape)

In [20]:
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train,y_train)
prediction = model.predict(X_test)
print("accruracy ",round(accuracy_score(y_test,prediction),4))
print(classification_report(y_test,prediction))
print("confusion matrix :- \n",confusion_matrix(y_test,prediction))

accruracy  0.7308
              precision    recall  f1-score   support

           0       0.58      0.64      0.61      1308
           1       0.82      0.77      0.79      2692

    accuracy                           0.73      4000
   macro avg       0.70      0.71      0.70      4000
weighted avg       0.74      0.73      0.73      4000

confusion matrix :- 
 [[ 841  467]
 [ 610 2082]]


In [21]:
# cross validations
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
scores = cross_val_score(model,X,y,cv=cv,scoring="accuracy")
print(scores)
print("Average accuracy ",round(scores.mean(),4))
print("Standard Deviation ",round(scores.std(),4))

[0.73175 0.72075 0.7305  0.7225  0.729  ]
Average accuracy  0.7269
Standard Deviation  0.0044


In [30]:
param_grid = {
    "criterion":["gini","entropy"],
    "max_depth":[3,5,7,10,15,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,6]
}
grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid.fit(X_train,y_train)
print("best parameters = ",grid.best_params_)
print("best cv score ",round(grid.best_score_,4))
grid_prediction = grid.predict(X_test)
print("Grid Search CV Test accuracy ",round(accuracy_score(y_test,grid_prediction),4))

best parameters =  {'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 10}
best cv score  0.8262
Grid Search CV Test accuracy  0.8427


In [33]:
# randomize search CV
param_dist = {
        "criterion":["gini","entropy"],
        "max_depth":randint(2,20),
        "min_samples_split":randint(2,15),
        "min_samples_leaf":randint(2,8)
}

random = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    random_state=42,
    scoring="accuracy",
    n_jobs=-1
)
random.fit(X_train,y_train)
print("best parameters = ",random.best_params_)
print("best cv score ",round(random.best_score_),4)
random_prediction = random.predict(X_test)
print("random search accuracy ",round(accuracy_score(y_test,random_prediction),4))

best parameters =  {'criterion': 'gini', 'max_depth': 8, 'min_samples_leaf': 3, 'min_samples_split': 4}
best cv score  1 4
random search accuracy  0.8413
